# 05 — Explainable AI & Error Analysis
 (ML Engineer / Explainable AI)

Uses `src/explain.py` to answer: **which features does the trained model
actually rely on, and where does it get things wrong?**

Every number below is computed from the trained model on real
(synthetic) data — nothing here is hand-picked.

In [ ]:
import sys
sys.path.append("../src")

import joblib
import pandas as pd
import matplotlib.pyplot as plt

from preprocessing import run_preprocessing
from explain import get_feature_names, global_feature_importance, built_in_feature_importance, SHAP_AVAILABLE
from evaluate import error_analysis

print("SHAP available in this environment:", SHAP_AVAILABLE)
print("(If False, this notebook automatically falls back to permutation")
print(" importance and the model's own built-in feature importances —")
print(" both are legitimate, model-derived explanation methods.)")

(X_train, X_test, y_job_train, y_job_test,
 y_track_train, y_track_test, preprocessor) = run_preprocessing()

job_model = joblib.load("../models/job_ready_model.joblib")
X_test_t = preprocessor.transform(X_test)
feature_names = get_feature_names(preprocessor)

## Global feature importance — built-in (Random Forest)

In [ ]:
built_in_imp = built_in_feature_importance(job_model, feature_names)
built_in_imp.head(15)

In [ ]:
top15 = built_in_imp.head(15)
plt.figure(figsize=(9, 6))
plt.barh(top15["feature"][::-1], top15["importance"][::-1], color="#4C72B0")
plt.title("Top 15 Features — Built-in Importance (Random Forest)")
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig("eda_assets_explainability_builtin.png", dpi=120) if False else None
plt.show()

## Global feature importance — permutation importance (model-agnostic, cross-check)

In [ ]:
perm_imp = global_feature_importance(job_model, X_test_t, y_job_test, feature_names, n_repeats=10)
perm_imp.head(15)

## Interpreting hackathon-related features specifically

Look at where `hackathons_attended`, `hackathons_won`,
`hackathon_win_rate`, and `finalist_status` rank relative to
`internship_months`, `end_to_end_projects`, `deployed_projects`, and the
interview/skill scores.

In [ ]:
hackathon_features = [
    "hackathons_attended", "hackathons_won", "hackathon_win_rate",
    "hackathon_finalist_rate", "finalist_status",
]
comparison_features = [
    "internship_months", "end_to_end_projects", "deployed_projects",
    "ml_score", "mock_interview_score", "avg_interview_readiness",
    "avg_core_skill_score", "real_world_exposure",
]

focus = perm_imp[perm_imp["feature"].isin(hackathon_features + comparison_features)]
focus.sort_values("importance_mean", ascending=False)

## Error analysis — the model's most confident mistakes

In [ ]:
X_test_readable = X_test.copy()
errors = error_analysis(job_model, X_test_t, y_job_test, X_test_readable, top_n=10)
cols_to_show = [
    "hackathons_attended", "hackathons_won", "internship_months",
    "end_to_end_projects", "deployed_projects", "ml_score",
    "mock_interview_score", "true_label", "predicted_label", "confidence",
]
errors[cols_to_show]

**What to look for:** do the mistakes cluster around students with an
unusual mix (e.g. very high hackathon count but no internship, or vice
versa)? That tells you where the model's blind spots are — useful to
mention in the viva when asked "what failed and what did you change?"

**This completes the modeling phase.** Next: the Streamlit dashboard
(`app/streamlit_app.py`, owned by Shilpi) wraps these saved models into
an interactive tool.